# Predicting Data Pipeline Failures

## Problem Statement

Data pipelines in large-scale Spark environments often fail due to factors such as:
- High data volume
- Transformation complexity (joins)
- Data skew
- Resource pressure

This project builds a machine learning model to predict pipeline failures before execution, enabling proactive monitoring and intervention.

## Technologies
- PySpark 
- Databricks
- MLflow 

## Dataset
- 100,000 simulated pipeline runs
- Features: workload size, partitions, joins, skew, execution time


# Step 1 — Data Generation

We simulate 100,000 pipeline runs with realistic characteristics.
Failure is determined by a combination of data volume, complexity, and skew —
mimicking real-world Spark pipeline behavior.

In [0]:
# To simulate real-world unpredictability, we add random noise
# to the failure score. This ensures the model cannot learn a
# perfect deterministic rule, reflecting the reality that
# pipelines sometimes fail for reasons beyond measurable features.

import random
from pyspark.sql.functions import current_timestamp, monotonically_increasing_id

random.seed(42)

def generate_pipeline_runs(n=100000):
    data = []
    for _ in range(n):
        rows = random.randint(1000, 100000)
        partitions = random.randint(10, 300)
        joins = random.randint(1, 15)
        skew = random.choice([0, 1])
        exec_time = random.randint(100, 1500)

        # Realistic failure logic WITH noise
        score = (
            0.00003 * rows +
            0.05 * joins +
            1.5 * skew +
            0.002 * exec_time
        )

        # Add randomness (real-world unpredictability)
        noise = random.uniform(-2, 2)
        score += noise

        failed = 1 if score > 5 else 0

        data.append((rows, partitions, joins, skew, exec_time, failed))
    return data

columns = [
    "rows_processed",
    "num_partitions",
    "num_joins",
    "skew_flag",
    "execution_time",
    "failed"
]

raw_data = generate_pipeline_runs(100000)
df       = spark.createDataFrame(raw_data, columns)


df = (
    df
    .withColumn("pipeline_id", monotonically_increasing_id())
    .withColumn("run_timestamp", current_timestamp())
    .repartition(50)
)


In [0]:
# Data generation sanity check 0 To check the class balance

df.groupBy("failed").count().show()

## Step 2 — Feature Engineering

Spark ML requires all input features to be combined into a single vector column.
We use VectorAssembler to transform individual columns into a unified `features` vector.

### Features used:
| Feature | Description |
|---------|------------|
| rows_processed | Volume of data in the pipeline run |
| num_partitions | Level of Spark parallelism |
| num_joins | Number of join operations (complexity) |
| skew_flag | Whether data skew was detected (0/1) |
| execution_time | Total runtime in seconds |

### Target:
| Column | Description |
|--------|------------|
| failed | 1 = failure, 0 = success |


In [0]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "rows_processed",
    "num_partitions",
    "num_joins",
    "skew_flag",
    "execution_time"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df_ml = assembler.transform(df).select("pipeline_id", "features", "failed")

In [0]:
df_ml.show(5, truncate=False)

## Step 3 — Train/Test Split

We split the dataset into:
- **80% training** — model learns patterns
- **20% testing** — model is evaluated on unseen data


In [0]:
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print("Train count:", train_df.count())
print("Test count:", test_df.count())


print("=== Train Distribution ===")
train_df.groupBy("failed").count().show()

print("=== Test Distribution ===")
test_df.groupBy("failed").count().show()

## Step 4 — Model Training

We train a Logistic Regression model for binary classification:
- 0 → pipeline success
- 1 → pipeline failure

Logistic Regression is chosen as a strong baseline model —
simple, interpretable, and effective for structured data.

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="failed",
    maxIter=10
)

model = lr.fit(train_df)


In [0]:
predictions = model.transform(test_df)

In [0]:
predictions.select(
    "pipeline_id",
    "failed",
    "prediction",
    "probability"
).show(10, truncate=False)

## Step 5 — Model Evaluation

We evaluate the model on the **test set only** (unseen data).

### Metrics used:
| Metric | What it measures |
|--------|-----------------|
| Accuracy | Overall correctness |
| AUC | How well model separates success vs failure |
| Precision | When model predicts failure, how often is it right? |
| Recall | Of all real failures, how many did the model catch? |

In [0]:
correct = predictions.filter("failed = prediction").count()
total = predictions.count()
accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="failed",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_evaluator.evaluate(predictions)

print(f"AUC: {auc:.4f}")

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Precision
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="failed",
    predictionCol="prediction",
    metricName="precisionByLabel",
    metricLabel=1.0
)
precision = precision_evaluator.evaluate(predictions)

# Recall
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="failed",
    predictionCol="prediction",
    metricName="recallByLabel",
    metricLabel=1.0
)
recall = recall_evaluator.evaluate(predictions)

print(f"Precision (failure class): {precision:.4f}")
print(f"Recall (failure class):    {recall:.4f}")

## Evaluation Analysis

| Metric | Value | Interpretation |
|--------|-------|---------------|
| Accuracy | 0.79 | Model is correct ~79% of the time |
| AUC | 0.87 | Strong separation between classes |
| Precision | 0.72 | 72% of predicted failures are real |
| Recall | 0.64 | Model catches 64% of actual failures |

### Key Observation

Recall is the weakest metric. In a pipeline monitoring scenario,
missed failures (false negatives) are more costly than false alarms.

### Potential improvements:
- Add more features (shuffle size, retry count, memory usage)
- Try stronger models (Random Forest, Gradient Boosted Trees)
- Adjust prediction threshold to favor recall over precision

## Step 7 — Experiment Tracking (MLflow)

We use MLflow to track:
- Model parameters
- Evaluation metrics
- Experiment metadata

This ensures reproducibility and enables comparison
across future model versions.

In [0]:
import mlflow

with mlflow.start_run():

    # Log parameters
    mlflow.log_param("model_type", "logistic_regression")
    mlflow.log_param("max_iter", 10)
    mlflow.log_param("train_test_split", "80/20")
    mlflow.log_param("dataset_size", 100000)
    mlflow.log_param("noise_range", "-2 to 2")

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)

    print("Experiment logged to MLflow")

## Step 8 — Real-World Prediction Demo

We simulate 3 new pipeline runs with different risk profiles
and use the trained model to predict failure probability.

This demonstrates how the model could be integrated into
a pipeline monitoring system for proactive alerting.

In [0]:
scenarios = [
    # Scenario 1: Safe pipeline
    (3000, 15, 2, 0, 150, "Low Risk"),

    # Scenario 2: Medium risk
    (25000, 100, 6, 0, 500, "Medium Risk"),

    # Scenario 3: Dangerous pipeline
    (80000, 250, 14, 1, 1300, "High Risk"),
]

scenario_data = [(r, p, j, s, e) for r, p, j, s, e, _ in scenarios]
scenario_labels = [label for *_, label in scenarios]

columns = [
    "rows_processed",
    "num_partitions",
    "num_joins",
    "skew_flag",
    "execution_time"
]

new_df = spark.createDataFrame(scenario_data, columns)

In [0]:
new_features = assembler.transform(new_df)
new_predictions = model.transform(new_features)

new_predictions.select(
    "rows_processed",
    "num_joins",
    "skew_flag",
    "prediction",
    "probability"
).show(truncate=False)

In [0]:
from pyspark.sql.functions import col, udf, round as spark_round
from pyspark.sql.types import FloatType

# Extract failure probability (second element)
get_failure_prob = udf(lambda v: float(v[1]), FloatType())

result = (
    new_predictions
    .withColumn("failure_probability", get_failure_prob(col("probability")))
    .select(
        "rows_processed",
        "num_partitions",
        "num_joins",
        "skew_flag",
        "execution_time",
        "prediction",
        "failure_probability"
    )
)

result.show(truncate=False)

## Conclusion

This project demonstrates an end-to-end machine learning workflow
in Databricks for predicting data pipeline failures.

### What was built:
- Large-scale synthetic dataset (100,000 pipeline runs)
- Feature engineering using Spark ML (VectorAssembler)
- Logistic Regression model with train/test evaluation
- MLflow experiment tracking
- Real-world prediction scenarios

### Key Results:
- AUC of 0.87 indicates strong class separation
- Recall of 0.64 highlights room for improvement in catching all failures
- Feature analysis shows **data skew** as the strongest failure predictor

### Future Improvements:
- Incorporate real pipeline logs (Spark metrics, shuffle stats)
- Test stronger models (Random Forest, GBT)
- Adjust classification threshold to optimize for recall
- Deploy model via MLflow Model Registry for batch inference

### Technologies Used:
- PySpark | Spark ML | Databricks | MLflow | Python

## Model Comparison — Random Forest

We now train a Random Forest classifier to compare against
Logistic Regression. Random Forest combines multiple decision trees,
each trained on different subsets of data, and uses majority voting
for the final prediction.

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="failed",
    numTrees=50,
    seed=42
)

rf_model = rf.fit(train_df)


In [0]:
rf_predictions = rf_model.transform(test_df)

rf_predictions.select(
    "pipeline_id",
    "failed",
    "prediction",
    "probability"
).show(10, truncate=False)

In [0]:
# Accuracy
rf_correct = rf_predictions.filter("failed = prediction").count()
rf_total = rf_predictions.count()
rf_accuracy = rf_correct / rf_total

# AUC
rf_auc = auc_evaluator.evaluate(rf_predictions)

# Precision
rf_precision = precision_evaluator.evaluate(rf_predictions)

# Recall
rf_recall = recall_evaluator.evaluate(rf_predictions)

print("=" * 40)
print("   RANDOM FOREST EVALUATION")
print("=" * 40)
print(f"   Accuracy:   {rf_accuracy:.4f}")
print(f"   AUC:        {rf_auc:.4f}")
print(f"   Precision:  {rf_precision:.4f}")
print(f"   Recall:     {rf_recall:.4f}")
print("=" * 40)

In [0]:
print("=" * 55)
print("         MODEL COMPARISON")
print("=" * 55)
print(f"{'Metric':<15} {'Logistic Reg':<15} {'Random Forest':<15}")
print("-" * 55)
print(f"{'Accuracy':<15} {accuracy:<15.4f} {rf_accuracy:<15.4f}")
print(f"{'AUC':<15} {auc:<15.4f} {rf_auc:<15.4f}")
print(f"{'Precision':<15} {precision:<15.4f} {rf_precision:<15.4f}")
print(f"{'Recall':<15} {recall:<15.4f} {rf_recall:<15.4f}")
print("=" * 55)

### Model Comparison Results

| Metric | Logistic Regression | Random Forest |
|--------|-------------------|---------------|
| Accuracy | 0.7868 | 0.7796 |
| AUC | 0.8691 | 0.8541 |
| Precision | 0.7150 | 0.7142 |
| Recall | 0.6419 | 0.6094 |

### Key Observation

Random Forest did NOT outperform Logistic Regression.

This is because the failure logic in the dataset follows a
**linear relationship** between features and outcome.
Logistic Regression, being a linear model, is naturally
better suited for this pattern.

### Lesson Learned

> A more complex model does not always mean better performance.
> The best model depends on the nature of the data — not
> the complexity of the algorithm.

This is a critical principle in applied machine learning.

In [0]:
with mlflow.start_run():

    mlflow.log_param("model_type", "random_forest")
    mlflow.log_param("num_trees", 50)
    mlflow.log_param("train_test_split", "80/20")
    mlflow.log_param("dataset_size", 100000)

    mlflow.log_metric("accuracy", rf_accuracy)
    mlflow.log_metric("auc", rf_auc)
    mlflow.log_metric("precision", rf_precision)
    mlflow.log_metric("recall", rf_recall)

    print("Random Forest experiment logged to MLflow")